# 07 · Flight-trajectory analytics (OpenSky data — research use only)

Detects unusual approaches (energy/descent-rate anomalies, go-arounds) from ADS-B trajectories with the open-source
`traffic` library. **License:** OpenSky Network data may be used for non-profit research/education, government purposes or
*internal* commercial testing only, may not be redistributed, and must be cited. This notebook therefore never writes
trajectory data into the repo — only aggregate metrics.

In [ ]:
!pip -q install traffic scikit-learn

In [ ]:
from traffic.data.samples import quickstart    # small sample bundled with the library for tutorials
import numpy as np
from sklearn.ensemble import IsolationForest
t = quickstart
feats, ids = [], []
for f in t:
    f = f.resample("5s")
    d = f.data
    if len(d) < 30 or "vertical_rate" not in d:
        continue
    last = d.tail(60)       # final ~5 minutes
    feats.append([last.vertical_rate.mean(), last.vertical_rate.std(), last.groundspeed.mean(), last.groundspeed.std(),
                  (last.altitude.diff() > 0).mean()])          # share of climbing samples ≈ go-around signal
    ids.append(f.flight_id or f.callsign)
X = np.nan_to_num(np.array(feats)); iso = IsolationForest(random_state=0).fit(X)
score = -iso.score_samples(X)
for i in np.argsort(-score)[:5]:
    print(ids[i], round(float(score[i]), 3))

For real analysis request historical data through OpenSky's research access (Trino) under their terms, keep it inside
your institution, and cite: Schäfer et al., "Bringing up OpenSky: A large-scale ADS-B sensor network for research", IPSN 2014.